# Multi-Modal Document Intelligence Agent
## Invoice & Form Processing — LangGraph Multi-Agent System

**Architecture:**
```
PDF/Image → Ingestion Pipeline (pdf2image + Pillow deskew)
         → Tesseract OCR (baseline text)
         → Extraction Agent (VLM: Claude vision)
         → Validation Agent (rule-based + ChromaDB/rapidfuzz)
         → Auto-Correction Agent (focused crop re-query)
         → SQLite Storage (full audit trail)
```

## 1. Install System Dependencies

In [ ]:
# Install Tesseract OCR + Poppler (needed by pdf2image)
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils libgl1
!tesseract --version

## 2. Install Python Packages

In [ ]:
!pip install -q \
    langgraph>=0.2.0 \
    langchain-core>=0.3.0 \
    pydantic>=2.0.0 \
    python-dotenv \
    pdf2image \
    Pillow \
    pytesseract \
    anthropic>=0.40.0 \
    opencv-python-headless \
    rapidfuzz \
    chromadb

print("All packages installed.")

## 3. Configuration & API Key

In [ ]:
import os
import getpass

# ── Option A: Colab Secrets (recommended) ──────────────────────────────────
# Add ANTHROPIC_API_KEY via Colab sidebar: 🔑 Secrets → New secret
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("API key loaded from Colab Secrets.")
except Exception:
    # ── Option B: Manual entry ────────────────────────────────────────────
    ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key: ")

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

# ── Settings ──────────────────────────────────────────────────────────────
VLM_BACKEND        = "claude"           # "claude" | "internvl" | "llava"
CLAUDE_MODEL       = "claude-sonnet-4-6"
OCR_DPI            = 300
TOTAL_TOLERANCE    = 0.01
FUZZY_THRESHOLD    = 80
MAX_CORRECTIONS    = 2
CROP_PADDING_PX    = 20
SQLITE_PATH        = "/content/invoices.db"
CHROMA_DIR         = "/content/chroma"
VENDOR_COLLECTION  = "vendors"

print(f"VLM Backend : {VLM_BACKEND}")
print(f"Claude Model: {CLAUDE_MODEL}")

## 4. Pydantic Schemas

In [ ]:
from __future__ import annotations
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field, field_validator


class LineItem(BaseModel):
    description: str
    quantity: Optional[float] = None
    unit_price: Optional[float] = None
    total: float


class ValidationStatus(str, Enum):
    VALID     = "valid"
    FAILED    = "failed"
    CORRECTED = "corrected"


class FieldValidation(BaseModel):
    field: str
    status: ValidationStatus
    message: Optional[str] = None
    original_value: Optional[str] = None
    corrected_value: Optional[str] = None


class InvoiceData(BaseModel):
    invoice_number: Optional[str] = None
    vendor_name:    Optional[str] = None
    invoice_date:   Optional[str] = None
    due_date:       Optional[str] = None
    subtotal:       Optional[float] = None
    tax_amount:     Optional[float] = None
    total_amount:   Optional[float] = None
    currency:       Optional[str] = "USD"
    line_items:     list[LineItem] = Field(default_factory=list)
    iban:           Optional[str] = None
    payment_terms:  Optional[str] = None

    @field_validator("total_amount", "subtotal", "tax_amount", mode="before")
    @classmethod
    def parse_amount(cls, v):
        if isinstance(v, str):
            cleaned = v.replace(",", "").replace("$", "").replace("€", "").strip()
            return float(cleaned) if cleaned else None
        return v


class ExtractionResult(BaseModel):
    raw_ocr_text:   str
    extracted_data: InvoiceData
    confidence:     float = Field(ge=0.0, le=1.0, default=0.0)
    vlm_response:   Optional[str] = None


class ValidationResult(BaseModel):
    is_valid:          bool
    field_validations: list[FieldValidation] = Field(default_factory=list)
    failed_fields:     list[str] = Field(default_factory=list)


class ProcessingRecord(BaseModel):
    document_id:         str
    source_path:         str
    raw_ocr_text:        str
    vlm_extraction:      dict
    corrections_applied: list[dict] = Field(default_factory=list)
    final_data:          dict
    validation_status:   ValidationStatus
    processing_notes:    list[str] = Field(default_factory=list)


print("Schemas loaded.")

## 5. Image Utilities (Deskew + Contrast)

In [ ]:
import io
import base64
import numpy as np
from PIL import Image, ImageEnhance


def deskew(img: Image.Image) -> Image.Image:
    try:
        import cv2
    except ImportError:
        return img
    gray = np.array(img.convert("L"))
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(binary > 0))
    if len(coords) < 10:
        return img
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = 90 + angle
    if abs(angle) < 0.5:
        return img
    return img.rotate(-angle, expand=True, fillcolor=(255, 255, 255))


def enhance_contrast(img: Image.Image, factor: float = 1.5) -> Image.Image:
    img = ImageEnhance.Contrast(img).enhance(factor)
    img = ImageEnhance.Sharpness(img).enhance(1.3)
    return img


def image_to_base64(img: Image.Image, fmt: str = "PNG") -> str:
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


print("Image utilities ready.")

## 6. Document Ingestion Pipeline

In [ ]:
from pathlib import Path
from typing import Union


def load_document(path: Union[str, Path]) -> list[Image.Image]:
    """Return preprocessed PIL images — one per page for PDFs."""
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return _pdf_to_images(path)
    elif suffix in {".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".webp"}:
        img = Image.open(path).convert("RGB")
        return [_preprocess(img)]
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


def _pdf_to_images(path: Path) -> list[Image.Image]:
    from pdf2image import convert_from_path
    pages = convert_from_path(str(path), dpi=OCR_DPI)
    return [_preprocess(p.convert("RGB")) for p in pages]


def _preprocess(img: Image.Image) -> Image.Image:
    return enhance_contrast(deskew(img))


print("Ingestion pipeline ready.")

## 7. Tesseract OCR

In [ ]:
import pytesseract
from pytesseract import Output


def ocr_extract_text(image: Image.Image) -> str:
    return pytesseract.image_to_string(image, config="--psm 6")


def ocr_extract_with_boxes(image: Image.Image) -> dict:
    return pytesseract.image_to_data(image, output_type=Output.DICT, config="--psm 6")


def crop_region(image: Image.Image, keyword: str, padding: int = CROP_PADDING_PX) -> Image.Image:
    """Crop image around the first bbox containing keyword; fall back to full image."""
    data = ocr_extract_with_boxes(image)
    for i, word in enumerate(data["text"]):
        if keyword.lower() in word.lower() and int(data["conf"][i]) > 0:
            x, y, w, h = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
            box = (
                max(0, x - padding), max(0, y - padding),
                min(image.width, x + w + padding), min(image.height, y + h + padding),
            )
            return image.crop(box)
    return image


print("OCR module ready.")

## 8. Vendor Matcher (ChromaDB + rapidfuzz)

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from rapidfuzz import fuzz, process as rfprocess

_chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
_ef = embedding_functions.DefaultEmbeddingFunction()
_vendor_col = _chroma_client.get_or_create_collection(
    name=VENDOR_COLLECTION, embedding_function=_ef
)


def add_vendors(vendors: list[str]) -> None:
    existing = set(_vendor_col.get()["documents"] or [])
    new = [v for v in vendors if v not in existing]
    if new:
        offset = len(existing)
        _vendor_col.add(
            documents=new,
            ids=[f"vendor_{offset + i}" for i in range(len(new))],
        )
    print(f"Added {len(new)} new vendor(s). Total: {_vendor_col.count()}")


def match_vendor(name: str) -> tuple[bool, str | None]:
    if _vendor_col.count() == 0:
        return True, name
    results = _vendor_col.query(query_texts=[name], n_results=min(3, _vendor_col.count()))
    candidates = results["documents"][0] if results["documents"] else []
    if not candidates:
        return False, None
    best, score, _ = rfprocess.extractOne(name, candidates, scorer=fuzz.token_sort_ratio)
    return (score >= FUZZY_THRESHOLD), (best if score >= FUZZY_THRESHOLD else None)


# Seed with a small set of example vendors — extend as needed
SEED_VENDORS = [
    "Acme Corp", "Global Supplies Inc", "Tech Solutions Ltd",
    "Office Depot", "Amazon Business", "Staples", "Dell Technologies",
]
add_vendors(SEED_VENDORS)
print("Vendor matcher ready.")

## 9. Extraction Agent (Claude Vision)

In [ ]:
import json
import re
import anthropic

_anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

_EXTRACTION_PROMPT = """You are an invoice data extraction expert.
Given the invoice image and its OCR text, extract the following fields into valid JSON:

{{
  "invoice_number": "string or null",
  "vendor_name": "string or null",
  "invoice_date": "YYYY-MM-DD or null",
  "due_date": "YYYY-MM-DD or null",
  "subtotal": number or null,
  "tax_amount": number or null,
  "total_amount": number or null,
  "currency": "3-letter ISO code",
  "iban": "string or null",
  "payment_terms": "string or null",
  "line_items": [
    {{"description": "...", "quantity": number, "unit_price": number, "total": number}}
  ]
}}

OCR TEXT:
{ocr_text}

Rules:
- Return ONLY the JSON object, no explanation.
- Use null for missing fields.
- Amounts must be plain numbers (no currency symbols).
- Dates must be ISO 8601 (YYYY-MM-DD)."""


def _parse_vlm_json(text: str) -> tuple[InvoiceData, float]:
    match = re.search(r"\{{[\s\S]*\}}", text)
    if not match:
        return InvoiceData(), 0.0
    try:
        data = json.loads(match.group())
        invoice = InvoiceData(**{k: v for k, v in data.items() if k in InvoiceData.model_fields})
        filled = sum(1 for v in invoice.model_dump().values() if v not in (None, [], {}))
        confidence = filled / len(InvoiceData.model_fields)
        return invoice, round(confidence, 2)
    except Exception as e:
        print(f"[parse error] {e}")
        return InvoiceData(), 0.0


def run_extraction(image: Image.Image, ocr_text: str) -> ExtractionResult:
    b64 = image_to_base64(image, fmt="PNG")
    prompt = _EXTRACTION_PROMPT.format(ocr_text=ocr_text[:4000])

    message = _anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=2048,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": b64}},
                {"type": "text", "text": prompt},
            ],
        }],
    )
    raw = message.content[0].text
    invoice_data, confidence = _parse_vlm_json(raw)
    return ExtractionResult(
        raw_ocr_text=ocr_text,
        extracted_data=invoice_data,
        confidence=confidence,
        vlm_response=raw,
    )


print("Extraction agent ready.")

## 10. Validation Agent

In [ ]:
from datetime import datetime


def _check_total(data: InvoiceData) -> list[FieldValidation]:
    if None in (data.subtotal, data.tax_amount, data.total_amount):
        return []
    expected = round(data.subtotal + data.tax_amount, 2)
    if abs(expected - data.total_amount) > TOTAL_TOLERANCE:
        return [FieldValidation(
            field="total_amount", status=ValidationStatus.FAILED,
            message=f"total {data.total_amount} != subtotal {data.subtotal} + tax {data.tax_amount} = {expected}",
        )]
    return [FieldValidation(field="total_amount", status=ValidationStatus.VALID)]


def _check_dates(data: InvoiceData) -> list[FieldValidation]:
    results = []
    for fname in ("invoice_date", "due_date"):
        val = getattr(data, fname)
        if val is None:
            continue
        try:
            datetime.strptime(val, "%Y-%m-%d")
            results.append(FieldValidation(field=fname, status=ValidationStatus.VALID))
        except ValueError:
            results.append(FieldValidation(
                field=fname, status=ValidationStatus.FAILED,
                message=f"Invalid date '{val}' (expected YYYY-MM-DD)",
            ))
    return results


def _check_iban(data: InvoiceData) -> list[FieldValidation]:
    if data.iban is None:
        return []
    iban = data.iban.replace(" ", "").upper()
    if re.match(r"^[A-Z]{2}[0-9]{2}[A-Z0-9]{11,30}$", iban):
        return [FieldValidation(field="iban", status=ValidationStatus.VALID)]
    return [FieldValidation(
        field="iban", status=ValidationStatus.FAILED,
        message=f"Invalid IBAN: {data.iban}",
    )]


def _check_vendor(data: InvoiceData) -> list[FieldValidation]:
    if not data.vendor_name:
        return [FieldValidation(field="vendor_name", status=ValidationStatus.FAILED,
                                message="Vendor name missing")]
    known, matched = match_vendor(data.vendor_name)
    if known and matched and matched.lower() != data.vendor_name.lower():
        return [FieldValidation(field="vendor_name", status=ValidationStatus.VALID,
                                message=f"Matched to: '{matched}'", corrected_value=matched)]
    return [FieldValidation(field="vendor_name", status=ValidationStatus.VALID)]


def run_validation(extraction: ExtractionResult) -> ValidationResult:
    data = extraction.extracted_data
    checks = _check_total(data) + _check_dates(data) + _check_iban(data) + _check_vendor(data)
    failed = [v.field for v in checks if v.status == ValidationStatus.FAILED]
    return ValidationResult(is_valid=len(failed) == 0, field_validations=checks, failed_fields=failed)


print("Validation agent ready.")

## 11. Auto-Correction Agent

In [ ]:
_CORRECTION_PROMPT = """The following field in an invoice failed validation: {field}

Current extracted value: {current_value}
Validation error: {error_message}

Look carefully at the image crop and re-extract ONLY "{field}".
Return a single JSON object: {{"field": "{field}", "value": <corrected value or null>}}

Rules: amounts = plain numbers, dates = YYYY-MM-DD, return ONLY JSON."""

_FIELD_KEYWORDS = {
    "total_amount": "total", "subtotal": "subtotal", "tax_amount": "tax",
    "invoice_date": "date", "due_date": "due", "invoice_number": "invoice",
    "vendor_name": "vendor", "iban": "iban",
}


def _correct_field(image: Image.Image, field: str, current: str, error: str):
    keyword = _FIELD_KEYWORDS.get(field, field.replace("_", " "))
    crop = crop_region(image, keyword)
    b64 = image_to_base64(crop, fmt="PNG")
    prompt = _CORRECTION_PROMPT.format(field=field, current_value=current, error_message=error)

    message = _anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": b64}},
                {"type": "text", "text": prompt},
            ],
        }],
    )
    raw = message.content[0].text
    m = re.search(r"\{{[\s\S]*?\}}", raw)
    if not m:
        return None
    try:
        return json.loads(m.group()).get("value")
    except Exception:
        return None


def run_correction(
    image: Image.Image,
    extraction: ExtractionResult,
    validation: ValidationResult,
) -> ExtractionResult:
    updated = extraction.extracted_data.model_copy(deep=True)
    for field in validation.failed_fields:
        fv = next((v for v in validation.field_validations if v.field == field), None)
        current = str(getattr(updated, field, ""))
        corrected = _correct_field(image, field, current, fv.message if fv else "")
        if corrected is not None:
            try:
                setattr(updated, field, corrected)
                print(f"  Corrected '{field}': {current!r} → {corrected!r}")
            except Exception:
                pass
    return ExtractionResult(
        raw_ocr_text=extraction.raw_ocr_text,
        extracted_data=updated,
        confidence=extraction.confidence,
        vlm_response=extraction.vlm_response,
    )


print("Correction agent ready.")

## 12. SQLite Storage

In [ ]:
import sqlite3


def _db() -> sqlite3.Connection:
    conn = sqlite3.connect(SQLITE_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db() -> None:
    with _db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS invoice_records (
                id               INTEGER PRIMARY KEY AUTOINCREMENT,
                document_id      TEXT UNIQUE NOT NULL,
                source_path      TEXT,
                raw_ocr_text     TEXT,
                vlm_extraction   TEXT,
                corrections      TEXT,
                final_data       TEXT,
                validation_status TEXT,
                processing_notes TEXT,
                created_at       TEXT DEFAULT (datetime('now'))
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS audit_log (
                id          INTEGER PRIMARY KEY AUTOINCREMENT,
                document_id TEXT NOT NULL,
                event       TEXT,
                details     TEXT,
                timestamp   TEXT DEFAULT (datetime('now'))
            )
        """)


def save_record(record: ProcessingRecord) -> None:
    with _db() as conn:
        conn.execute("""
            INSERT INTO invoice_records
                (document_id, source_path, raw_ocr_text, vlm_extraction,
                 corrections, final_data, validation_status, processing_notes)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(document_id) DO UPDATE SET
                final_data=excluded.final_data,
                validation_status=excluded.validation_status,
                corrections=excluded.corrections
        """, (
            record.document_id, record.source_path, record.raw_ocr_text,
            json.dumps(record.vlm_extraction), json.dumps(record.corrections_applied),
            json.dumps(record.final_data), record.validation_status.value,
            json.dumps(record.processing_notes),
        ))


def log_event(doc_id: str, event: str, details: dict = None) -> None:
    with _db() as conn:
        conn.execute(
            "INSERT INTO audit_log (document_id, event, details) VALUES (?, ?, ?)",
            (doc_id, event, json.dumps(details or {})),
        )


def list_records(limit: int = 50) -> list[dict]:
    with _db() as conn:
        rows = conn.execute(
            "SELECT * FROM invoice_records ORDER BY created_at DESC LIMIT ?", (limit,)
        ).fetchall()
    return [dict(r) for r in rows]


init_db()
print(f"SQLite DB initialised at {SQLITE_PATH}")

## 13. LangGraph Workflow

In [ ]:
import uuid
from typing import TypedDict
from langgraph.graph import END, StateGraph


class InvoiceState(TypedDict):
    document_id:        str
    source_path:        str
    images:             list
    ocr_texts:          list[str]
    extraction:         object
    validation:         object
    correction_attempts: int
    final_status:       object
    processing_notes:   list[str]


def node_ingest(state: InvoiceState) -> InvoiceState:
    images = load_document(state["source_path"])
    state["images"] = images
    state["processing_notes"].append(f"Ingested {len(images)} page(s)")
    log_event(state["document_id"], "ingested", {"pages": len(images)})
    return state


def node_ocr(state: InvoiceState) -> InvoiceState:
    texts = [ocr_extract_text(img) for img in state["images"]]
    state["ocr_texts"] = texts
    state["processing_notes"].append("OCR complete")
    log_event(state["document_id"], "ocr_complete", {"chars": sum(len(t) for t in texts)})
    return state


def node_extract(state: InvoiceState) -> InvoiceState:
    combined = "\n\n--- PAGE BREAK ---\n\n".join(state["ocr_texts"])
    result = run_extraction(state["images"][0], combined)
    state["extraction"] = result
    state["processing_notes"].append(f"Extracted (confidence={result.confidence:.2f})")
    log_event(state["document_id"], "extraction_complete", {"confidence": result.confidence})
    return state


def node_validate(state: InvoiceState) -> InvoiceState:
    result = run_validation(state["extraction"])
    state["validation"] = result
    msg = "Validation passed" if result.is_valid else f"Failed on: {result.failed_fields}"
    state["processing_notes"].append(msg)
    log_event(state["document_id"], "validation", {"valid": result.is_valid, "failed": result.failed_fields})
    return state


def node_correct(state: InvoiceState) -> InvoiceState:
    state["correction_attempts"] += 1
    updated = run_correction(state["images"][0], state["extraction"], state["validation"])
    state["extraction"] = updated
    state["processing_notes"].append(f"Correction attempt {state['correction_attempts']}")
    log_event(state["document_id"], "correction", {"attempt": state["correction_attempts"]})
    return state


def node_store(state: InvoiceState) -> InvoiceState:
    v = state["validation"]
    if v and v.is_valid:
        status = ValidationStatus.VALID
    elif state["correction_attempts"] > 0:
        status = ValidationStatus.CORRECTED
    else:
        status = ValidationStatus.FAILED
    state["final_status"] = status

    corrections = [fv.model_dump() for fv in (v.field_validations if v else [])
                   if fv.status == ValidationStatus.CORRECTED]

    save_record(ProcessingRecord(
        document_id=state["document_id"],
        source_path=state["source_path"],
        raw_ocr_text="\n\n".join(state["ocr_texts"]),
        vlm_extraction=state["extraction"].extracted_data.model_dump() if state["extraction"] else {},
        corrections_applied=corrections,
        final_data=state["extraction"].extracted_data.model_dump() if state["extraction"] else {},
        validation_status=status,
        processing_notes=state["processing_notes"],
    ))
    log_event(state["document_id"], "stored", {"status": status.value})
    return state


def _route(state: InvoiceState) -> str:
    if state["validation"] and not state["validation"].is_valid:
        if state["correction_attempts"] < MAX_CORRECTIONS:
            return "correct"
    return "store"


def build_graph():
    g = StateGraph(InvoiceState)
    g.add_node("ingest",   node_ingest)
    g.add_node("ocr",      node_ocr)
    g.add_node("extract",  node_extract)
    g.add_node("validate", node_validate)
    g.add_node("correct",  node_correct)
    g.add_node("store",    node_store)
    g.set_entry_point("ingest")
    g.add_edge("ingest",   "ocr")
    g.add_edge("ocr",      "extract")
    g.add_edge("extract",  "validate")
    g.add_conditional_edges("validate", _route, {"correct": "correct", "store": "store"})
    g.add_edge("correct",  "validate")
    g.add_edge("store",    END)
    return g.compile()


GRAPH = build_graph()
print("LangGraph workflow compiled.")

## 14. Upload & Process Invoice

In [ ]:
from google.colab import files

print("Upload an invoice PDF or image file (PNG/JPG/TIFF):")
uploaded = files.upload()

for filename in uploaded:
    filepath = f"/content/{filename}"
    with open(filepath, "wb") as f:
        f.write(uploaded[filename])
    print(f"\nSaved: {filepath}")

In [ ]:
# Process the uploaded file (change filename if you uploaded multiple)
INPUT_FILE = filepath  # or set manually: INPUT_FILE = "/content/invoice.pdf"

initial_state: InvoiceState = {
    "document_id":        str(uuid.uuid4()),
    "source_path":        INPUT_FILE,
    "images":             [],
    "ocr_texts":          [],
    "extraction":         None,
    "validation":         None,
    "correction_attempts": 0,
    "final_status":       None,
    "processing_notes":   [],
}

print(f"Processing: {INPUT_FILE}\n")
result_state = GRAPH.invoke(initial_state)
print("\nDone!")

## 15. Display Results

In [ ]:
import pandas as pd
from IPython.display import display

data = result_state["extraction"].extracted_data
validation = result_state["validation"]

print("=" * 55)
print(f"  STATUS      : {result_state['final_status'].value.upper()}")
print(f"  Document ID : {result_state['document_id']}")
print(f"  Corrections : {result_state['correction_attempts']}")
print("=" * 55)

summary = {
    "Field": ["Vendor", "Invoice #", "Invoice Date", "Due Date",
               "Currency", "Subtotal", "Tax", "Total", "IBAN", "Payment Terms"],
    "Value": [
        data.vendor_name, data.invoice_number, data.invoice_date, data.due_date,
        data.currency, data.subtotal, data.tax_amount, data.total_amount,
        data.iban, data.payment_terms,
    ],
}
display(pd.DataFrame(summary))

if data.line_items:
    print("\nLine Items:")
    display(pd.DataFrame([li.model_dump() for li in data.line_items]))

if validation and validation.field_validations:
    print("\nValidation Details:")
    vdf = pd.DataFrame([v.model_dump() for v in validation.field_validations])
    display(vdf[["field", "status", "message"]])

print("\nProcessing Log:")
for note in result_state["processing_notes"]:
    print(f"  • {note}")

## 16. Show Preprocessed Image

In [ ]:
import matplotlib.pyplot as plt

images = result_state["images"]
n = len(images)
fig, axes = plt.subplots(1, n, figsize=(8 * n, 10))
if n == 1:
    axes = [axes]
for i, (ax, img) in enumerate(zip(axes, images)):
    ax.imshow(img)
    ax.set_title(f"Page {i+1} (preprocessed)")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 17. Browse All Stored Records

In [ ]:
records = list_records()
print(f"{len(records)} record(s) in database.")
if records:
    df = pd.DataFrame(records)[["document_id", "source_path", "validation_status", "created_at"]]
    display(df)

## 18. Download SQLite Database

In [ ]:
from google.colab import files
files.download(SQLITE_PATH)
print("Downloaded invoices.db")

## 19. (Optional) Add Your Own Vendors to ChromaDB

In [ ]:
MY_VENDORS = [
    # Add your known vendor names here
    "Your Vendor Name",
    "Another Vendor Ltd",
]
add_vendors(MY_VENDORS)